# EMA Pullback (9-21) on SPY
## Strategy Brief
The EMA Pullback (9-21) strategy is a trend-following approach that uses two exponential moving averages (EMAs) to identify potential buy signals. The strategy looks for a pullback in price when the 9-day EMA crosses above the 21-day EMA, indicating a potential upward trend. Trades are entered when this crossover occurs, and positions are held until the EMAs cross in the opposite direction. Historical testing suggests this strategy can capture significant trends, though it may underperform in sideways markets.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters for our EMA Pullback strategy. These include the short and long EMA periods and the ticker symbol for SPY.

In [ ]:
SHORT_EMA_PERIOD = 9
LONG_EMA_PERIOD = 21
TICKER = 'SPY'
START_DATE = '2010-01-01'

## PHASE 2 - Data Exploration
We will download historical price data for SPY from Yahoo Finance, calculate the 9-day and 21-day EMAs, and plot them over the price to visualize potential trading signals.

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE)

# Calculate EMAs
data['EMA_9'] = data['Close'].ewm(span=SHORT_EMA_PERIOD, adjust=False).mean()
data['EMA_21'] = data['Close'].ewm(span=LONG_EMA_PERIOD, adjust=False).mean()

# Plot
data[['Close', 'EMA_9', 'EMA_21']].plot(figsize=(14, 7))
plt.title('SPY Price with 9 and 21 Day EMAs')
plt.show()

## PHASE 3 - Strategy Engineering
We will create a signal series based on the crossover of the EMAs. A buy signal is generated when the 9-day EMA crosses above the 21-day EMA.

In [ ]:
data['Signal'] = 0
# Buy signal
data.loc[data['EMA_9'] > data['EMA_21'], 'Signal'] = 1
# Sell signal
data.loc[data['EMA_9'] < data['EMA_21'], 'Signal'] = -1

# Positions
data['Position'] = data['Signal'].shift(1)


## PHASE 4 - Coding & Backtesting
We will backtest the strategy by calculating daily returns and plotting the equity curve.

In [ ]:
data['Market_Return'] = data['Close'].pct_change()
data['Strategy_Return'] = data['Market_Return'] * data['Position']
data['Equity_Curve'] = (1 + data['Strategy_Return']).cumprod()

# Plot equity curve
data['Equity_Curve'].plot(figsize=(14, 7))
plt.title('Equity Curve of EMA Pullback Strategy')
plt.show()

## PHASE 5 - Performance Evaluation
We will evaluate the strategy's performance using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown, and compare these to a buy-and-hold strategy.

In [ ]:
import numpy as np

# Calculate performance metrics
cagr = (data['Equity_Curve'].iloc[-1]) ** (1 / ((data.index[-1] - data.index[0]).days / 365.25)) - 1
sharpe_ratio = (data['Strategy_Return'].mean() / data['Strategy_Return'].std()) * np.sqrt(252)
sortino_ratio = (data['Strategy_Return'].mean() / data[data['Strategy_Return'] < 0]['Strategy_Return'].std()) * np.sqrt(252)
max_drawdown = (data['Equity_Curve'].cummax() - data['Equity_Curve']).max() / data['Equity_Curve'].cummax().max()
calmar_ratio = cagr / max_drawdown

# Buy-and-hold comparison
bh_cagr = (data['Close'].iloc[-1] / data['Close'].iloc[0]) ** (1 / ((data.index[-1] - data.index[0]).days / 365.25)) - 1
bh_max_drawdown = (data['Close'].cummax() - data['Close']).max() / data['Close'].cummax().max()

# Print results
print(f"CAGR: {cagr:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Sortino Ratio: {sortino_ratio:.2f}")
print(f"Calmar Ratio: {calmar_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%}")

print(f"\nBuy-and-Hold CAGR: {bh_cagr:.2%}")
print(f"Buy-and-Hold Max Drawdown: {bh_max_drawdown:.2%}")

## PHASE 6 - Deploy & Monitor
We will create a function to download the last 60 days of SPY data, calculate today's signal, and print the current position.

In [ ]:
def check_current_signal():
    recent_data = yf.download(TICKER, period='60d')
    recent_data['EMA_9'] = recent_data['Close'].ewm(span=SHORT_EMA_PERIOD, adjust=False).mean()
    recent_data['EMA_21'] = recent_data['Close'].ewm(span=LONG_EMA_PERIOD, adjust=False).mean()
    recent_data['Signal'] = 0
    recent_data.loc[recent_data['EMA_9'] > recent_data['EMA_21'], 'Signal'] = 1
    recent_data.loc[recent_data['EMA_9'] < recent_data['EMA_21'], 'Signal'] = -1
    current_signal = recent_data['Signal'].iloc[-1]
    position = 'Long' if current_signal == 1 else 'Short' if current_signal == -1 else 'Neutral'
    print(f"Current position: {position}")

check_current_signal()